# 01 — EGFR Bioactivity Data Collection

## Project
Machine Learning-Guided Virtual Screening for EGFR Inhibitors

## Objective
Retrieve experimentally measured bioactivity data for compounds tested against human EGFR from the ChEMBL database.

## Why this matters
These experimentally characterized compounds will form the training dataset for a machine-learning QSAR model that predicts whether new compounds are likely to inhibit EGFR.

## Imports

In [1]:
import pandas as pd
import requests
from pathlib import Path

## Set up project paths

In [2]:
# Get the project root directory
PROJECT_ROOT = Path.cwd().parent

# Define data folders
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Make sure they exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

Project root: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery
Raw data directory: /Users/gracescanlan/Desktop/egfr-ml-drug-discovery/data/raw


## Test the ChEMBL API

In [3]:
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

url = f"{BASE_URL}/target.json"

params = {
    "limit": 5
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


## Search for EGFR

In [4]:
url = f"{BASE_URL}/target/search.json"

params = {
    "q": "epidermal growth factor receptor"
}

response = requests.get(url, params=params)

egfr_search = response.json()

In [5]:
egfr_search.keys()

dict_keys(['page_meta', 'targets'])

## Turn the search results into a table

In [6]:
targets_df = pd.DataFrame(egfr_search["targets"])

targets_df.head()

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Mus musculus,Epidermal growth factor receptor,33.0,False,CHEMBL3608,"[{'accession': 'Q01279', 'component_descriptio...",SINGLE PROTEIN,10090
1,[],Homo sapiens,Pro-epidermal growth factor,33.0,False,CHEMBL5734,"[{'accession': 'P01133', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,Growth factor receptor-bound protein 7,31.0,False,CHEMBL1649051,"[{'accession': 'Q14451', 'component_descriptio...",SINGLE PROTEIN,9606
3,[],Homo sapiens,Delta and Notch-like epidermal growth factor-r...,30.0,False,CHEMBL5291567,"[{'accession': 'Q8NFT8', 'component_descriptio...",SINGLE PROTEIN,9606
4,[],Mus musculus,Protein cereblon/Epidermal growth factor receptor,30.0,False,CHEMBL6193842,"[{'accession': 'Q01279', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,10090


## Look at useful target information

In [7]:
targets_df.columns.tolist()

['cross_references',
 'organism',
 'pref_name',
 'score',
 'species_group_flag',
 'target_chembl_id',
 'target_components',
 'target_type',
 'tax_id']

In [8]:
targets_df[
    [
        "target_chembl_id",
        "pref_name",
        "target_type",
        "organism"
    ]
].head(20)

,target_chembl_id,pref_name,target_type,organism
0,CHEMBL3608,Epidermal growth factor receptor,SINGLE PROTEIN,Mus musculus
1,CHEMBL5734,Pro-epidermal growth factor,SINGLE PROTEIN,Homo sapiens
2,CHEMBL1649051,Growth factor receptor-bound protein 7,SINGLE PROTEIN,Homo sapiens
3,CHEMBL5291567,Delta and Notch-like epidermal growth factor-r...,SINGLE PROTEIN,Homo sapiens
4,CHEMBL6193842,Protein cereblon/Epidermal growth factor receptor,PROTEIN-PROTEIN INTERACTION,Mus musculus
5,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,Homo sapiens
6,CHEMBL4295760,Epidermal growth factor receptor substrate 15,SINGLE PROTEIN,Homo sapiens
7,CHEMBL2363049,Epidermal growth factor receptor,PROTEIN FAMILY,Homo sapiens
8,CHEMBL3712972,Epidermal growth factor-like protein 7,SINGLE PROTEIN,Homo sapiens
9,CHEMBL3713025,Protein Cripto,SINGLE PROTEIN,Homo sapiens


In [9]:
human_egfr = targets_df[
    targets_df["organism"].eq("Homo sapiens")
][
    [
        "target_chembl_id",
        "pref_name",
        "target_type",
        "organism"
    ]
]

human_egfr

,target_chembl_id,pref_name,target_type,organism
1,CHEMBL5734,Pro-epidermal growth factor,SINGLE PROTEIN,Homo sapiens
2,CHEMBL1649051,Growth factor receptor-bound protein 7,SINGLE PROTEIN,Homo sapiens
3,CHEMBL5291567,Delta and Notch-like epidermal growth factor-r...,SINGLE PROTEIN,Homo sapiens
5,CHEMBL203,Epidermal growth factor receptor,SINGLE PROTEIN,Homo sapiens
6,CHEMBL4295760,Epidermal growth factor receptor substrate 15,SINGLE PROTEIN,Homo sapiens
7,CHEMBL2363049,Epidermal growth factor receptor,PROTEIN FAMILY,Homo sapiens
8,CHEMBL3712972,Epidermal growth factor-like protein 7,SINGLE PROTEIN,Homo sapiens
9,CHEMBL3713025,Protein Cripto,SINGLE PROTEIN,Homo sapiens
10,CHEMBL4523680,Protein cereblon/Epidermal growth factor receptor,PROTEIN-PROTEIN INTERACTION,Homo sapiens
11,CHEMBL6193830,UBR/Epidermal growth factor receptor,PROTEIN-PROTEIN INTERACTION,Homo sapiens


## Confirm Human EGFR Target

The ChEMBL target search identified CHEMBL203 as the human epidermal growth factor receptor (EGFR), classified as a single protein target.

This target will be used to retrieve experimental bioactivity measurements for compounds tested against EGFR.

In [10]:
EGFR_TARGET_ID = "CHEMBL203"

target_url = f"{BASE_URL}/target/{EGFR_TARGET_ID}.json"

response = requests.get(target_url)
response.raise_for_status()

egfr_target = response.json()

print("ChEMBL ID:", egfr_target["target_chembl_id"])
print("Target name:", egfr_target["pref_name"])
print("Target type:", egfr_target["target_type"])
print("Organism:", egfr_target["organism"])

ChEMBL ID: CHEMBL203
Target name: Epidermal growth factor receptor
Target type: SINGLE PROTEIN
Organism: Homo sapiens


## Retrieve EGFR Bioactivity Records

ChEMBL stores experimental measurements describing the activity of compounds against biological targets.

For this project, IC50 measurements will ultimately be used to represent EGFR inhibitory activity.

In [11]:
activity_url = f"{BASE_URL}/activity.json"

params = {
    "target_chembl_id": EGFR_TARGET_ID,
    "limit": 100
}

response = requests.get(activity_url, params=params)
response.raise_for_status()

activity_data = response.json()

In [12]:
activity_data.keys()

dict_keys(['activities', 'page_meta'])

In [13]:
len(activity_data["activities"])

100

## Look at the total number of EGFR activity records

In [14]:
activity_data["page_meta"]

{'limit': 100,
 'next': '/chembl/api/data/activity.json?limit=100&offset=100&target_chembl_id=CHEMBL203',
 'offset': 0,
 'previous': None,
 'total_count': 58847}

In [15]:
total_activities = activity_data["page_meta"]["total_count"]

print(f"Total EGFR activity records in ChEMBL: {total_activities:,}")

Total EGFR activity records in ChEMBL: 58,847


## Turn first 100 records into a DataFrame

In [16]:
activity_sample = pd.DataFrame(activity_data["activities"])

print("Shape:", activity_sample.shape)

activity_sample.head()

Shape: (100, 47)


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,32260,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,None,0.041
1,None,NaN,32263,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,None,0.3
2,None,NaN,32265,[],CHEMBL615325,Inhibition of ligand-induced proliferation in ...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,None,7.82
3,None,NaN,32267,[],CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,None,0.17
4,None,NaN,32270,[],CHEMBL621151,Inhibition of autophosphorylation of human epi...,F,None,None,BAO_0000190,...,Homo sapiens,Epidermal growth factor receptor,9606,NaN,None,IC50,uM,UO_0000065,None,0.04


In [17]:
activity_sample.columns.tolist()

['action_type',
 'activity_comment',
 'activity_id',
 'activity_properties',
 'assay_chembl_id',
 'assay_description',
 'assay_type',
 'assay_variant_accession',
 'assay_variant_mutation',
 'bao_endpoint',
 'bao_format',
 'bao_label',
 'canonical_smiles',
 'data_validity_comment',
 'data_validity_description',
 'document_chembl_id',
 'document_journal',
 'document_year',
 'ligand_efficiency',
 'modality',
 'molecule_chembl_id',
 'molecule_pref_name',
 'parent_molecule_chembl_id',
 'pchembl_value',
 'potential_duplicate',
 'qudt_units',
 'record_id',
 'relation',
 'src_id',
 'standard_flag',
 'standard_relation',
 'standard_text_value',
 'standard_type',
 'standard_units',
 'standard_upper_value',
 'standard_value',
 'target_chembl_id',
 'target_organism',
 'target_pref_name',
 'target_tax_id',
 'text_value',
 'toid',
 'type',
 'units',
 'uo_units',
 'upper_value',
 'value']

In [18]:
columns_to_view = [
    "molecule_chembl_id",
    "standard_type",
    "standard_relation",
    "standard_value",
    "standard_units",
    "pchembl_value",
    "assay_chembl_id",
    "assay_type",
    "target_chembl_id"
]

activity_sample[columns_to_view].head(20)

,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value,assay_chembl_id,assay_type,target_chembl_id
0,CHEMBL68920,IC50,=,41.0,nM,7.39,CHEMBL674637,B,CHEMBL203
1,CHEMBL68920,IC50,=,300.0,nM,6.52,CHEMBL621151,F,CHEMBL203
2,CHEMBL68920,IC50,=,7820.0,nM,5.11,CHEMBL615325,F,CHEMBL203
3,CHEMBL69960,IC50,=,170.0,nM,6.77,CHEMBL674637,B,CHEMBL203
4,CHEMBL69960,IC50,=,40.0,nM,7.40,CHEMBL621151,F,CHEMBL203
5,CHEMBL69960,IC50,=,440.0,nM,6.36,CHEMBL615325,F,CHEMBL203
6,CHEMBL137635,IC50,=,9300.0,nM,5.03,CHEMBL677833,B,CHEMBL203
7,CHEMBL306988,IC50,=,500000.0,nM,NaN,CHEMBL674643,B,CHEMBL203
8,CHEMBL306988,Ki,NaN,NaN,nM,NaN,CHEMBL675639,B,CHEMBL203
9,CHEMBL66879,IC50,=,3000000.0,nM,NaN,CHEMBL674643,B,CHEMBL203


In [19]:
activity_sample["standard_type"].value_counts().head(20)

standard_type
IC50    69
Ki      31
Name: count, dtype: int64

## Inspect IC50

In [20]:
ic50_sample = activity_sample[
    activity_sample["standard_type"] == "IC50"
].copy()

print("IC50 records in first 100:", len(ic50_sample))

ic50_sample[
    [
        "molecule_chembl_id",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value"
    ]
].head(20)

IC50 records in first 100: 69


,molecule_chembl_id,standard_relation,standard_value,standard_units,pchembl_value
0,CHEMBL68920,=,41.0,nM,7.39
1,CHEMBL68920,=,300.0,nM,6.52
2,CHEMBL68920,=,7820.0,nM,5.11
3,CHEMBL69960,=,170.0,nM,6.77
4,CHEMBL69960,=,40.0,nM,7.40
5,CHEMBL69960,=,440.0,nM,6.36
6,CHEMBL137635,=,9300.0,nM,5.03
7,CHEMBL306988,=,500000.0,nM,NaN
9,CHEMBL66879,=,3000000.0,nM,NaN
11,CHEMBL77085,=,96000.0,nM,4.02


## Retrieve Full EGFR IC50 Dataset

The initial ChEMBL query identified multiple types of bioactivity measurements, including IC50 and Ki. To maintain a consistent definition of inhibitory activity, this analysis will focus on IC50 measurements.

IC50 represents the concentration of a compound required to reduce the measured biological activity by 50%. Lower IC50 values therefore generally indicate greater inhibitory potency.

The following query retrieves all available IC50 activity records associated with the human EGFR target (CHEMBL203).

## Query only IC50

In [21]:
ic50_url = f"{BASE_URL}/activity.json"

ic50_params = {
    "target_chembl_id": EGFR_TARGET_ID,
    "standard_type": "IC50",
    "limit": 1000
}

response = requests.get(ic50_url, params=ic50_params)
response.raise_for_status()

ic50_data = response.json()

print("Total EGFR IC50 records:",
      f"{ic50_data['page_meta']['total_count']:,}")

Total EGFR IC50 records: 26,600


In [22]:
print("Records downloaded:", len(ic50_data["activities"]))
print("Total records available:", ic50_data["page_meta"]["total_count"])
print("Next page:", ic50_data["page_meta"]["next"])

Records downloaded: 1000
Total records available: 26600
Next page: /chembl/api/data/activity.json?limit=1000&offset=1000&target_chembl_id=CHEMBL203&standard_type=IC50


## Assess IC50 Data Quality

Before downloading the complete dataset, the initial 1,000 IC50 records were examined to determine the availability and consistency of standardized activity values, units, and measurement relationships.

This assessment will guide filtering criteria for the final dataset.

In [23]:
sample_ic50_df = pd.DataFrame(ic50_data["activities"])

print("Sample shape:", sample_ic50_df.shape)

Sample shape: (1000, 47)


In [24]:
sample_ic50_df["standard_units"].value_counts(dropna=False)

standard_units
nM    1000
Name: count, dtype: int64

In [25]:
sample_ic50_df["standard_relation"].value_counts(dropna=False)

standard_relation
=      780
>      199
NaN     21
Name: count, dtype: int64

In [26]:
columns_to_check = [
    "molecule_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "assay_chembl_id"
]

sample_ic50_df[columns_to_check].isna().sum()

molecule_chembl_id      0
standard_value         21
standard_units          0
standard_relation      21
pchembl_value         275
assay_chembl_id         0
dtype: int64

In [27]:
sample_ic50_df["assay_type"].value_counts(dropna=False)

assay_type
B    803
F    197
Name: count, dtype: int64

In [28]:
sample_ic50_df["data_validity_comment"].value_counts(dropna=False)

data_validity_comment
NaN                      939
Outside typical range     61
Name: count, dtype: int64

In [29]:
sample_ic50_df["potential_duplicate"].value_counts(dropna=False)

potential_duplicate
0    909
1     91
Name: count, dtype: int64

## Define Initial Inclusion Criteria

To construct a consistent dataset for QSAR modeling, bioactivity records were restricted to exact IC50 measurements from biochemical binding assays.

Initial inclusion criteria:

- Human EGFR target (CHEMBL203)
- IC50 bioactivity measurement
- Standardized units of nM
- Exact measurements (`standard_relation = "="`)
- Binding assay (`assay_type = "B"`)

Additional data-quality filtering and consolidation of repeated compound measurements will be performed during preprocessing.

In [30]:
filtered_params = {
    "target_chembl_id": EGFR_TARGET_ID,
    "standard_type": "IC50",
    "standard_units": "nM",
    "standard_relation": "=",
    "assay_type": "B",
    "limit": 1
}

response = requests.get(
    ic50_url,
    params=filtered_params,
    timeout=30
)

response.raise_for_status()

filtered_count_data = response.json()

print(
    "Records meeting initial criteria:",
    f"{filtered_count_data['page_meta']['total_count']:,}"
)

Records meeting initial criteria: 18,381


## Download Filtered EGFR IC50 Dataset

After applying the initial inclusion criteria, 18,381 EGFR IC50 activity records remained eligible for retrieval.

The complete filtered dataset was downloaded in paginated batches from ChEMBL. A maximum-record safeguard was implemented to prevent unintended or indefinite API requests.

In [31]:
import time

MAX_RECORDS = 20_000
PAGE_SIZE = 1000

all_filtered_records = []

url = ic50_url

download_params = {
    "target_chembl_id": EGFR_TARGET_ID,
    "standard_type": "IC50",
    "standard_units": "nM",
    "standard_relation": "=",
    "assay_type": "B",
    "limit": PAGE_SIZE
}

while url and len(all_filtered_records) < MAX_RECORDS:

    response = requests.get(
        url,
        params=download_params,
        timeout=30
    )

    response.raise_for_status()

    page_data = response.json()

    records = page_data["activities"]

    all_filtered_records.extend(records)

    print(
        f"Downloaded "
        f"{len(all_filtered_records):,} / "
        f"{filtered_count_data['page_meta']['total_count']:,}"
    )

    next_page = page_data["page_meta"]["next"]

    if next_page is None:
        break

    url = f"https://www.ebi.ac.uk{next_page}"

    # The next-page URL already contains the query parameters
    download_params = None

    time.sleep(0.2)

Downloaded 1,000 / 18,381
Downloaded 2,000 / 18,381
Downloaded 3,000 / 18,381
Downloaded 4,000 / 18,381
Downloaded 5,000 / 18,381
Downloaded 6,000 / 18,381
Downloaded 7,000 / 18,381
Downloaded 8,000 / 18,381
Downloaded 9,000 / 18,381
Downloaded 10,000 / 18,381
Downloaded 11,000 / 18,381
Downloaded 12,000 / 18,381
Downloaded 13,000 / 18,381
Downloaded 14,000 / 18,381
Downloaded 15,000 / 18,381
Downloaded 16,000 / 18,381
Downloaded 17,000 / 18,381
Downloaded 18,000 / 18,381
Downloaded 18,381 / 18,381


In [32]:
filtered_ic50_df = pd.DataFrame(all_filtered_records)

print("Downloaded records:", len(filtered_ic50_df))
print("Expected records:",
      filtered_count_data["page_meta"]["total_count"])

filtered_ic50_df.shape

Downloaded records: 18381
Expected records: 18381


(18381, 47)

In [33]:
filtered_ic50_df[
    [
        "molecule_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "assay_type"
    ]
].head(10)

,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,assay_type
0,CHEMBL68920,IC50,=,41.0,nM,B
1,CHEMBL69960,IC50,=,170.0,nM,B
2,CHEMBL137635,IC50,=,9300.0,nM,B
3,CHEMBL306988,IC50,=,500000.0,nM,B
4,CHEMBL66879,IC50,=,3000000.0,nM,B
5,CHEMBL77085,IC50,=,96000.0,nM,B
6,CHEMBL443268,IC50,=,5310.0,nM,B
7,CHEMBL76979,IC50,=,264000.0,nM,B
8,CHEMBL76589,IC50,=,125.0,nM,B
9,CHEMBL76904,IC50,=,35000.0,nM,B
